# ODS 8

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
import json
import requests
from functools import reduce
path = "../datos"
#path = "/content"

## Mejor y Peor ingreso 2024


In [209]:
df = pd.read_excel(path + "/8.5.1A_sh_03_es.xls", sheet_name=0, header = None)
df = df.iloc[8:-6] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[:, [1, 3,4]]
df.columns = ["Entidad","Hombres", "Mujeres"]
mejor_ing_hombre = df['Hombres']
mejor_ing_mujer = df['Mujeres']
mejor_ing_hombre = mejor_ing_hombre.values.tolist()[0]
mejor_ing_mujer = mejor_ing_mujer.values.tolist()[0]


mejor_ing_hombre
mejor_ing_mujer

72.91

In [212]:
df = pd.read_excel(path + "/8.5.1A_sh_07_es.xls", sheet_name=0, header = None)
df = df.iloc[8:-6] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[:, [1, 3,4]]
df.columns = ["Entidad","Hombres", "Mujeres"]
peor_ing_hombre = df['Hombres']
peor_ing_mujer = df['Mujeres']
peor_ing_hombre = peor_ing_hombre.values.tolist()[0]
peor_ing_mujer = peor_ing_mujer.values.tolist()[0]


peor_ing_hombre
peor_ing_mujer

36.32

## Hombres y mujeres informales

In [ ]:
df = pd.read_excel(path + "/8.3.1_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-12] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df.columns = ["Año","Total", "Hombres", "Mujeres"]
df = df[["Año", "Mujeres", "Hombres"]]
df = df.sort_values(by=['Año'])

# 1. DATOS SIMULADOS (3 años, 2 categorías)
anios = df['Año'].tolist()
total_bloques = 100

# Valores para cada año [Mujer, Hombre]
datos_evolucion = {
    '2024': [6, 9],
    '2025': [10, 12],
    '2026': [8, 14]
}

# 2. FUNCIÓN PARA CREAR LAS BARRAS DE UN AÑO ESPECÍFICO
def crear_trazos(anio):
    aux, val_m, val_h = df.loc[df['Año'] == anio].values.tolist()[0]

    # Trazo Mujeres
    trace_m = go.Bar(
        y=['Mujer'], x=[val_m],
        orientation='h',
        marker=dict(color='#f06292', line=dict(color='white', width=4)),
        width=0.5,
        name='Mujer',
        hovertemplate=f"Año {anio}<br>Porcentaje: {val_m}<extra></extra>"
    )

    # Trazo Hombres
    trace_h = go.Bar(
        y=['Hombre'], x=[val_h],
        orientation='h',
        marker=dict(color='#42a5f5', line=dict(color='white', width=4)),
        width=0.5,
        name='Hombre',
        hovertemplate=f"Año {anio}<br>Porcentaje: {val_h}<extra></extra>"
    )

    # Fondo gris para completar los 15 bloques
    fondo_m = go.Bar(y=['Mujer'], x=[total_bloques], orientation='h',
                     marker=dict(color='#fce4ec'), width=0.5, showlegend=False, hoverinfo='skip')
    fondo_h = go.Bar(y=['Hombre'], x=[total_bloques], orientation='h',
                     marker=dict(color='#e1f5fe'), width=0.5, showlegend=False, hoverinfo='skip')

    return [fondo_m, fondo_h, trace_m, trace_h]

# 3. CONFIGURACIÓN INICIAL (Año 2024)
fig = go.Figure(
    data=crear_trazos(2024),
    layout=go.Layout(
        title="% de Hombres y Mujeres en la informalidad<br><sup>Fuentes: INEGI, ENOEN 2015-2024</sup>",
        barmode='overlay', # Superponemos el color al fondo gris
        plot_bgcolor='white',
        xaxis=dict(showticklabels=False,range=[0, total_bloques], showgrid=False, zeroline=False, dtick=1),
        yaxis=dict(showgrid=False),
        updatemenus=[{
            "type": "buttons",
            "buttons": [{"label": "▶", "method": "animate", "args": [None, {"frame": {"duration": 500}}]}]
        }]
    )
)

# 4. CREACIÓN DE FRAMES (La magia del tiempo)
fig.frames = [go.Frame(data=crear_trazos(yr), name=yr) for yr in anios]

# 5. CONFIGURACIÓN DEL SLIDER
sliders = [{
    "active": 0,
    "currentvalue": {"visible": False},
    "steps": [{"label": yr, "method": "animate", "args": [[yr], {"frame": {"duration": 300, "redraw": True}}]} for yr in anios]
}]

fig.update_layout(sliders=sliders,
                  margin=dict(b=100))

# Truco visual: Añadir líneas blancas verticales para simular los cortes de los bloques
for i in range(total_bloques + 1):
    fig.add_vline(x=i-0.5, line_width=3, line_color="white")

fig.show()

## Niños que trabajan

In [ ]:
df = pd.read_excel(path + "/8.7.1_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[8:-13] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[:, [1, 2, 3]]
df.columns = ["Entidad Federativa", "Total", "Total_niños"]
df['Total'] = df['Total'].apply(pd.to_numeric, errors='coerce').astype(float)
df['Total_niños'] = df['Total_niños'].apply(pd.to_numeric, errors='coerce').astype(float)
nacional_val = df.loc[df['Entidad Federativa'] == "Estados Unidos Mexicanos"].values.tolist()[0][2]
top_3 = df.nlargest(3, 'Total_niños', keep='all')
df = top_3

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df['Entidad Federativa'],
    y=df['Total_niños'],
    marker_color='#64bec0',
    name='Entidad Federativa',
    hovertemplate=(
        "<b>Estado:</b> %{x}<br>" +
        "<b>Proporción:</b> %{y:.2f}%<br>" +
        "<extra></extra>"
    )
))

# 3. Agregar la Línea Nacional (Horizontal)
fig.add_hline(
    y=nacional_val,
    line_dash="dash", # Línea punteada
    line_color="red",
    line_width=3,
    annotation_text=f"Promedio Nacional: {nacional_val:.2f} " + "%",
    annotation_position="top right",
    annotation_font=dict(color="red", size=14)
)

# 4. Estética del Layout
fig.update_layout(
    title="% Niños de 5 a 14 años trabajando <br> <sup>INEGI ENOE 2022</sup>",
    xaxis_title="Los 3 estados con mayor proporción",
    yaxis_title="Porcentaje (%)",
    plot_bgcolor='white',
    height=500
)

# Añadir rejilla suave en el eje Y para mejor lectura
fig.update_yaxes(showgrid=True, gridcolor='lightgray')

fig.show()

## NINIS por nivel educativo

In [ ]:
df = pd.read_excel(path + "/8.6.1A_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[9:-39] # Eliminar todos los renglones de texto, solo dejar el nacional
#df.columns = df.iloc[0] # El encabezado es el primer renglón
#df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
cols = [1,4, 6, 7, 46, 48, 49, 88, 90, 91, 104, 106, 107]
df = df.iloc[:, cols]
df.columns = ["Entidad", "Primaria2024", "Secundaria2024", "Preparatoria2024", "Primaria2022", "Secundaria2022", "Preparatoria2022", "Primaria2020", "Secundaria2020", "Preparatoria2020", "Primaria2018", "Secundaria2018", "Preparatoria2018"]
df_long = df.melt(
    id_vars=['Entidad'],
    var_name='Variable',
    value_name='Valor'
)

# 3. Separamos el Nivel y el Año (extrayendo los últimos 4 caracteres para el año)
df_long['Año'] = df_long['Variable'].str[-4:]
df_long['Nivel'] = df_long['Variable'].str[:-4]

# Limpiamos las columnas innecesarias y ordenamos
df_long = df_long[['Entidad', 'Año', 'Nivel', 'Valor']].sort_values(['Año'], ascending=[True])
df_long['Valor'] = df_long['Valor'].apply(pd.to_numeric, errors='coerce').astype(float)

colores = {
    "Primaria": "#bb6b00",
    "Secundaria": "#690500",
    "Preparatoria": "#9C4A3C"
}
fig = go.Figure()

# Para que no se apilen y queden una al lado de la otra con el mismo color por Nivel:
for nivel in df_long['Nivel'].unique():
    df_filtrado = df_long[df_long['Nivel'] == nivel]

    fig.add_trace(go.Bar(
        x=[df_filtrado['Nivel'], df_filtrado['Año']], # Eje X multi-nivel
        y=df_filtrado['Valor'],
        name=nivel,
        marker_color=colores.get(nivel, '#333'), # Asigna color por categoría
        text=df_filtrado['Valor'].round(1),      # Muestra el dato sobre la barra
        textposition='auto',
        customdata=df_filtrado['Año'],
        hovertemplate=(
            "<b>Nivel:</b> " + nivel + "<br>" +
            "<b>Año:</b> %{customdata}<br>" +
            "<b>Porcentaje:</b> %{y:.2f}%<br>" +
            "<extra></extra>" # Esto oculta el nombre del trace a la derecha
        )
    ))

# Configuración visual Pro
fig.update_layout(
    title={
        'text': "<b>% nacional jóvenes (de 15 a 24 años) que NO estudian, <br> NO tienen empleo NI reciben capacitación<br> y que solamente concluyeron la...</b><br><sup>INEGI ENOE 2018-2024</sup><span style='font-size:12px'>  </span>",
        'y':0.95, 'x':0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Categoría / Año",
    yaxis_title="Porcentaje (%)",
    barmode='group',
    template="plotly_white", # Fondo blanco limpio
    bargap=0.15,            # Espacio entre grupos
    bargroupgap=0.1,        # Espacio entre barras del mismo grupo
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.8,
        xanchor="right",
        x=1
    )
)

# Mejorar el formato de los números en el eje Y
fig.update_yaxes(gridcolor='LightGrey', zerolinecolor='Black')

fig.show()